# 📐 Fundamentos: K-Nearest Neighbors (KNN)

## La idea, en una frase
> **Para predecir algo nuevo, busca los casos más parecidos que ya conoces y promedia su respuesta.**

Si quieres saber cuánto rinde un auto que no has visto, buscas los **k autos más parecidos** del conjunto de entrenamiento y promedias su rendimiento. Si quieres saber si un pasajero sobrevivió, buscas los **k pasajeros más parecidos** y votas.

## Qué lo hace distinto de lo que ya viste
| | Regresión lineal / logística | Árbol | KNN |
|---|---|---|---|
| ¿Qué aprende? | Coeficientes (una fórmula) | Reglas de corte | **Nada**: se guarda los datos |
| ¿Cuándo trabaja? | Al entrenar | Al entrenar | **Al predecir** |
| ¿Asume una forma? | Sí, una recta o una curva logística | No | **No**, se adapta a cualquier forma |
| ¿Se puede interpretar? | Sí, cada coeficiente dice algo | Sí, se dibuja | Poco: no hay coeficientes |
| ¿Necesita escalado? | No | No | **Sí, obligatorio** |

Por eso se le llama un modelo **perezoso** (*lazy learner*): no construye una fórmula, sólo memoriza y compara.

## 1. ¿Qué significa "parecido"? La distancia euclidiana

Es el teorema de Pitágoras extendido a todas las variables. Entre dos autos con medidas $(x_1, x_2, \dots)$:

$$d = \sqrt{(x_1 - x_1')^2 + (x_2 - x_2')^2 + \dots}$$

Mientras más chica la distancia, más parecidos son.

In [ ]:
import numpy as np

def distancia_euclidiana(a, b):
    """Distancia en línea recta entre dos observaciones."""
    return np.sqrt(np.sum((np.array(a) - np.array(b)) ** 2))

# Dos autos descritos por [peso (lb), año del modelo]
auto_A = [2200, 78]
auto_B = [2400, 80]

print("Diferencia en peso:", abs(auto_A[0] - auto_B[0]), "lb")
print("Diferencia en año: ", abs(auto_A[1] - auto_B[1]), "años")
print("Distancia:", round(distancia_euclidiana(auto_A, auto_B), 2))

⚠️ **Mira el resultado: la distancia es prácticamente la diferencia de peso.** El año aportó 2 unidades y el peso aportó 200, así que el peso **aplasta** al año en el cálculo.

No es que el peso importe más; es que está medido en una escala más grande. Si midiéramos el peso en toneladas en vez de libras, la distancia cambiaría por completo y KNN elegiría otros vecinos.

> ### 🔑 Por eso **KNN siempre necesita que las variables estén escaladas**.

Con `StandardScaler` todas quedan con media 0 y desviación 1, así cada variable aporta lo mismo a la distancia. Esto **no** era necesario en regresión lineal, ni en logística, ni en los árboles de la sesión pasada: ahí escalar no cambiaba nada. Aquí cambia el modelo entero.

## 2. El hiperparámetro k

`k` es cuántos vecinos se consultan antes de responder:

| k | Qué pasa | Riesgo |
|---|---|---|
| **k = 1** | Copia la respuesta del vecino más cercano | **Sobreajuste**: memoriza el ruido |
| **k mediano** | Promedia unos pocos casos parecidos | El punto ideal |
| **k muy grande** | Promedia casi todo el dataset | **Subajuste**: predice casi lo mismo siempre |

Con `k = 1` el error en entrenamiento es **cero** (cada punto es su propio vecino), pero en datos nuevos falla. Es la trampa clásica de este modelo.

El valor de `k` se elige con **validación cruzada**, nunca mirando el conjunto de prueba.

In [ ]:
import plotly.express as px
from sklearn.neighbors import KNeighborsRegressor

# Datos de juguete: una relación con forma de onda, más ruido
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 120)
y = np.sin(x) * 3 + rng.normal(0, 0.6, 120)

datos = {'x': x, 'valor real': y}
for k in (1, 5, 40):
    datos[f'k = {k}'] = KNeighborsRegressor(n_neighbors=k).fit(x.reshape(-1, 1), y).predict(x.reshape(-1, 1))

fig = px.line(datos, x='x', y=[c for c in datos if c != 'x'],
              title='Efecto de k: de memorizar el ruido a aplanarlo todo',
              template='plotly_white', labels={'value': 'y', 'variable': ''})
fig.update_traces(selector={'name': 'valor real'}, mode='markers', marker=dict(size=5, color='lightgray'))
fig.update_layout(width=950, height=500)
fig.show()

✅ **Qué observar:**
- **k = 1** (línea quebrada) pasa por **todos** los puntos, incluido el ruido
- **k = 5** sigue la forma de la onda sin perseguir cada punto: es el equilibrio
- **k = 40** aplana la curva: promedia tantos vecinos que pierde la señal

## 3. Lo que hay que recordar
1. KNN **no entrena**, compara: guarda los datos y mide distancias al predecir
2. **Escalar es obligatorio**, porque la distancia depende de las unidades
3. **k chico sobreajusta, k grande subajusta**; se elige con validación cruzada
4. No da coeficientes, así que **se gana en flexibilidad y se pierde en interpretación**

Ahora llévalo a los dos casos que ya conoces: `caso_mpg.ipynb` y `caso_titanic.ipynb`.